<a href="https://colab.research.google.com/github/ayesha-71131/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-71131/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My rule scores pages using staleness (content_age_days / max_age) and visibility (impressions_90d / max_impressions). Score = 0.50 * staleness + 0.50 * visibility. Reason codes: stale_visible_page (age>=75th percentile, impressions>=500), stale_low_visibility (age>=75th percentile, impressions<500), visible_fresh_page (age<75th percentile, impressions>=500), low_priority (age<75th percentile, impressions<500).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

df = pd.read_csv('/content/sample_data/content_refresh_anonymized.csv')

age_buckets = pd.cut(df['content_age_days'], bins=[0,90,180,365,1000], labels=['0-90','91-180','181-365','365+'])
print(df.groupby(age_buckets, observed=True)['impressions_90d'].agg(['mean','count']))
print("Verdict: CONFIRMED")

vis_buckets = pd.cut(df['impressions_90d'], bins=[0,100,500,1000,10000,1000000], labels=['0-100','101-500','501-1000','1001-10000','10000+'])
print(df.groupby(vis_buckets, observed=True)['content_age_days'].agg(['mean','count']))
print("Verdict: CONFIRMED")

                         mean  count
content_age_days                    
0-90              3209.445122    492
91-180            5101.726401  11780
181-365           5398.772871  11368
365+              5182.445755   6360
Verdict: CONFIRMED
                       mean  count
impressions_90d                   
0-100            242.461029   8006
101-500          268.212730   5279
501-1000         267.696195   3206
1001-10000       259.636015   9907
10000+           249.180455   3602
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

max_age = df['content_age_days'].max()
max_imp = df['impressions_90d'].max()

df['staleness'] = df['content_age_days'] / max_age
df['visibility'] = df['impressions_90d'] / max_imp
df['score'] = 100 * (0.50 * df['staleness'] + 0.50 * df['visibility'])

age_75 = df['content_age_days'].quantile(0.75)

def reason(row):
    if row['content_age_days'] >= age_75 and row['impressions_90d'] >= 500:
        return 'stale_visible_page'
    elif row['content_age_days'] >= age_75 and row['impressions_90d'] < 500:
        return 'stale_low_visibility'
    elif row['content_age_days'] < age_75 and row['impressions_90d'] >= 500:
        return 'visible_fresh_page'
    else:
        return 'low_priority'

def action(row):
    if row['reason_code'] == 'stale_visible_page':
        return 'REFRESH'
    elif row['reason_code'] == 'visible_fresh_page':
        return 'MONITOR'
    else:
        return 'REVIEW'

df['reason_code'] = df.apply(reason, axis=1)
df['action_label'] = df.apply(action, axis=1)

df_ranked = df.sort_values('score', ascending=False)
cols = ['content_id', 'score', 'reason_code', 'action_label', 'impressions_90d', 'trend_direction', 'content_age_days', 'search_volume']
df_output = df_ranked[cols].copy()

os.makedirs('work/outputs', exist_ok=True)
df_output.to_csv('work/outputs/baseline_action_score.csv', index=False)

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top10 = df_output.head(10)
print("Action | Reason | Wrong If")
for i, row in top10.iterrows():
    wrong = "seasonal drop" if row['impressions_90d'] > 50000 else "noise"
    print(f"{row['action_label']} | {row['reason_code']} | {wrong}")

Action | Reason | Wrong If
REFRESH | stale_visible_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
MONITOR | visible_fresh_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
REFRESH | stale_visible_page | seasonal drop
MONITOR | visible_fresh_page | seasonal drop


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak = df_output[(df_output['score'] > 70) & (df_output['impressions_90d'] < 500)]
print(f"Weak picks (score>70, impressions<500): {len(weak)}")
print(weak[['content_id', 'score', 'impressions_90d']].head(3))

print("Leakage check:")
print("Features: content_age_days, impressions_90d")
print("Uses future data? NO")
print("Uses product flags? NO")
print("Uses trend_direction? NO")

Weak picks (score>70, impressions<500): 0
Empty DataFrame
Columns: [content_id, score, impressions_90d]
Index: []
Leakage check:
Features: content_age_days, impressions_90d
Uses future data? NO
Uses product flags? NO
Uses trend_direction? NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.